# Fase 2 · Normalización del corpus (spaCy + POS)

Este notebook construye `data/scientific_normalized_corpus.tsv` a partir de `data/scientific_raw_corpus.tsv`, añadiendo las columnas `title_norm` y `abstract_norm`.

**Objetivo:** normalizar título y abstract de los 500 documentos con la función `normalize()` aprobada en la prueba (`02_prueba_normalizacion.ipynb`), que se reutilizará **sin cambios** para las consultas.

## Decisiones de diseño

| Decisión | Valor | Motivo |
|---|---|---|
| Herramienta | spaCy, modelo `en_core_web_sm` | Tokenización, POS y lematización en inglés en un solo pipeline |
| Tokenización | La de spaCy | Maneja siglas, guiones y abreviaturas mejor que separar por espacios |
| Stopwords | Se eliminan por POS: `DET, ADP, CCONJ, SCONJ, PRON` | Son palabras gramaticales; lo pide el enunciado |
| Auxiliares | También se elimina `AUX` (*be, have, can, will*) | Decisión del equipo: no aportan tema y *be* aparecería en casi todos los documentos |
| Se conservan | `NOUN, PROPN, VERB, ADJ` y el resto de etiquetas | Indican de qué trata el documento |
| Puntuación y espacios | Se eliminan | No aportan contenido al modelo vectorial |
| Lematización | `token.lemma_` | Agrupa variantes (*models* → *model*, *trained* → *train*) |
| Minúsculas | Sí, después de lematizar | *Model* y *model* deben ser el mismo término |
| Números, guiones, términos técnicos | Se dejan como los produce spaCy | Decisión aprobada en la prueba; los casos raros se documentan en la sección 6 |
| Componentes desactivados | Solo `ner` | El reconocimiento de entidades no interviene en POS ni lemas. El `parser` **se mantiene**: sin él cambian algunas etiquetas (*which* DET↔PRON, *remains* AUX→VERB) y el resultado ya no coincide con la prueba aprobada (se verifica en la sección 3) |


## 1. Configuración

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd
import spacy

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_PATH = ROOT / "data" / "scientific_raw_corpus.tsv"
NORM_PATH = ROOT / "data" / "scientific_normalized_corpus.tsv"
REPORT_PATH = ROOT / "results" / "normalization_stats.md"
SPACY_MODEL = "en_core_web_sm"
BATCH_SIZE = 64

nlp = spacy.load(SPACY_MODEL, disable=["ner"])
print("spaCy", spacy.__version__, "| modelo", nlp.meta["name"], nlp.meta["version"])
print("Pipeline activo:", nlp.pipe_names)

spaCy 3.8.16 | modelo core_web_sm 3.8.0
Pipeline activo: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer']


## 2. Función de normalización

Es la misma que se aprobó en la prueba. `normalize_doc` trabaja sobre un `Doc` ya procesado (para usar `nlp.pipe` por lotes) y `normalize` es la entrada para un texto suelto, como una consulta.

In [2]:
# Etiquetas POS (universales de spaCy) que se consideran stopwords y se eliminan.
# Son palabras gramaticales: sirven para armar la frase, aparecen en casi todos
# los textos y no indican el tema del documento.
STOP_POS = {
    "DET",    # determinantes: the, a, an, this, each
    "ADP",    # preposiciones: of, in, on, with, from
    "CCONJ",  # conjunciones coordinantes: and, or, but
    "SCONJ",  # conjunciones subordinantes: that, because, if, while
    "PRON",   # pronombres: we, it, they, which
    "AUX",    # verbos auxiliares: be, have, can, will (decisión del equipo, no la pide el enunciado)
}
# Se conservan NOUN, PROPN, VERB y ADJ, que son las que dicen de qué trata el texto.


def normalize_doc(doc) -> str:
    return " ".join(
        t.lemma_.lower() for t in doc
        if not (t.is_punct or t.is_space) and t.pos_ not in STOP_POS
    )


def normalize(text: str) -> str:
    """Normaliza un texto suelto (documento o consulta)."""
    return normalize_doc(nlp(text))


normalize("The models were trained on large datasets of scientific papers.")

'model train large dataset scientific paper'

## 3. Verificación contra la prueba aprobada

Se desactivó `ner` para ir más rápido. Aquí se comprueba que eso **no cambia** el resultado: los 5 documentos de la prueba deben quedar idénticos a como salen con el pipeline completo.

In [3]:
raw = pd.read_csv(RAW_PATH, sep="\t", dtype=str, keep_default_na=False)
print(raw.shape)

nlp_full = spacy.load(SPACY_MODEL)
sample = raw["abstract"].head(5)
assert [normalize(s) for s in sample] == [normalize_doc(nlp_full(s)) for s in sample], \
    "Desactivar ner cambió la normalización"
del nlp_full
print("OK: mismo resultado que el pipeline completo en los 5 documentos de la prueba")

(500, 10)


OK: mismo resultado que el pipeline completo en los 5 documentos de la prueba


## 4. Normalización de todo el corpus

Se procesan títulos y abstracts por lotes con `nlp.pipe`. De paso se cuentan los tokens y las palabras eliminadas para el reporte.

In [4]:
removed = Counter()
tokens_before, tokens_after = {}, {}


def normalize_column(texts: pd.Series, field: str) -> list[str]:
    out, before = [], []
    for doc in nlp.pipe(texts, batch_size=BATCH_SIZE):
        removed.update(t.lemma_.lower() for t in doc if t.pos_ in STOP_POS)
        before.append(sum(1 for t in doc if not (t.is_punct or t.is_space)))
        out.append(normalize_doc(doc))
    tokens_before[field] = before
    tokens_after[field] = [len(s.split()) for s in out]
    return out


raw["title_norm"] = normalize_column(raw["title"], "title")
raw["abstract_norm"] = normalize_column(raw["abstract"], "abstract")
raw[["arxiv_id", "title", "title_norm"]].head()

,arxiv_id,title,title_norm
0,2610.00840,Contextual trajectory and incremental contextu...,contextual trajectory incremental contextual d...
1,2610.00833,VERITYGATE: A Four-Gate Schema-Level Faithfuln...,veritygate four gate schema level faithfulness...
2,2610.00827,Verbalized and Internal Probabilities Are Coup...,verbalize internal probabilities couple large ...
3,2610.06903,Component and Dimension Sparsity in Transforme...,component dimension sparsity transformer refus...
4,2610.00795,Can large language models unlock discrete data...,large language model unlock discrete datum oph...


## 5. Construcción y validación

Criterios de aceptación de la Fase 2:
- mismas 500 filas y mismo orden de `arxiv_id` que el corpus crudo,
- ningún `title_norm` ni `abstract_norm` vacío,
- sin tabuladores ni saltos de línea en los campos normalizados,
- la consulta se normaliza con **la misma función** que el corpus.

In [5]:
COLUMNS = ["arxiv_id", "primary_category", "title", "abstract", "title_norm", "abstract_norm"]
norm = raw[COLUMNS].copy()

assert len(norm) == len(raw) and norm["arxiv_id"].is_unique
assert (norm["title_norm"].str.len() > 0).all(), "Hay títulos normalizados vacíos"
assert (norm["abstract_norm"].str.len() > 0).all(), "Hay abstracts normalizados vacíos"
assert not norm[["title_norm", "abstract_norm"]].apply(lambda c: c.str.contains(r"[\t\n\r]")).any().any()

# Corpus y consulta usan la misma función: normalizar de nuevo un título debe dar lo mismo
i = 0
assert normalize(norm.loc[i, "title"]) == norm.loc[i, "title_norm"]
print(f"{len(norm)} documentos válidos")
print("Consulta de ejemplo:", normalize("misinformation detection using transformers"))

500 documentos válidos
Consulta de ejemplo: misinformation detection use transformer


## 6. Guardado del TSV y verificación de carga

In [6]:
norm.to_csv(NORM_PATH, sep="\t", index=False)

reloaded = pd.read_csv(NORM_PATH, sep="\t", dtype=str, keep_default_na=False)
assert reloaded.shape == norm.shape
assert (reloaded["arxiv_id"] == norm["arxiv_id"]).all()
assert (reloaded["abstract_norm"] == norm["abstract_norm"]).all()
print(NORM_PATH, reloaded.shape)

/home/mrangello/Documentos/PracticaML/data/scientific_normalized_corpus.tsv (500, 6)


## 7. Reporte de normalización

Tokens antes y después, reducción por categoría, tamaño de vocabulario y palabras eliminadas más frecuentes. Se guarda también en `results/normalization_stats.md`.

In [7]:
stats = pd.DataFrame({
    "tokens_titulo_antes": tokens_before["title"],
    "tokens_titulo_despues": tokens_after["title"],
    "tokens_abstract_antes": tokens_before["abstract"],
    "tokens_abstract_despues": tokens_after["abstract"],
    "primary_category": norm["primary_category"],
})

totals = stats.drop(columns="primary_category").sum()
reduction = pd.DataFrame({
    "tokens_antes": [totals["tokens_titulo_antes"], totals["tokens_abstract_antes"]],
    "tokens_despues": [totals["tokens_titulo_despues"], totals["tokens_abstract_despues"]],
}, index=["title", "abstract"])
reduction["reduccion_%"] = (100 * (1 - reduction["tokens_despues"] / reduction["tokens_antes"])).round(1)

by_cat = stats.groupby("primary_category")[["tokens_abstract_antes", "tokens_abstract_despues"]].sum()
by_cat["reduccion_%"] = (100 * (1 - by_cat["tokens_abstract_despues"] / by_cat["tokens_abstract_antes"])).round(1)

vocab = pd.DataFrame({
    "vocabulario": [len(set(" ".join(norm[c]).split())) for c in ("title_norm", "abstract_norm")],
}, index=["title_norm", "abstract_norm"])

top_removed = pd.DataFrame(removed.most_common(20), columns=["lema", "veces"])
display(reduction, by_cat, vocab, top_removed)

,tokens_antes,tokens_despues,reduccion_%
title,5417,4336,20.0
abstract,103630,73532,29.0


,tokens_abstract_antes,tokens_abstract_despues,reduccion_%
primary_category,,,
cs.CL,25303,17833,29.5
cs.CV,26274,18895,28.1
cs.IR,26624,19001,28.6
cs.LG,25429,17803,30.0


,vocabulario
title_norm,1863
abstract_norm,7958


,lema,veces
0,the,3638
1,and,3427
2,a,2563
3,of,1636
4,be,1219
5,we,1142
6,on,1141
7,in,1127
8,for,1111
9,that,1081


In [8]:
from datetime import datetime, timezone

report = f"""# Normalización del corpus

- Generado: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}
- spaCy {spacy.__version__}, modelo `{nlp.meta['name']}` {nlp.meta['version']}
- POS eliminadas: {', '.join(sorted(STOP_POS))}; también puntuación y espacios
- Lematización + minúsculas
- Documentos: {len(norm)}

## Reducción de tokens
{reduction.to_markdown()}

## Reducción en abstracts por categoría
{by_cat.to_markdown()}

## Tamaño de vocabulario
{vocab.to_markdown()}

## Palabras eliminadas más frecuentes
{top_removed.to_markdown(index=False)}
"""
REPORT_PATH.write_text(report, encoding="utf-8")
print(REPORT_PATH)

/home/mrangello/Documentos/PracticaML/results/normalization_stats.md


## 8. Resultado

Archivos generados:
- `data/scientific_normalized_corpus.tsv`: corpus con `title_norm` y `abstract_norm`, `document_id = arxiv_id`.
- `results/normalization_stats.md`: reporte de normalización.

**Casos conocidos (de la prueba), documentar en el reporte:**
- `to` (`PART`) no está en `STOP_POS` y se conserva.
- Los auxiliares (`AUX`) se eliminan; algunos verbos plenos pueden etiquetarse como `AUX` y perderse (p. ej. *remains*).
- El etiquetador puede fallar con términos técnicos (p. ej. *RoBERTa* como `ADJ`).
- Las siglas en plural no se lematizan (*LLMs* → `llms`, *LLM* → `llm`).

**Siguiente paso:** Fase 3, representación VSM (12 configuraciones).